In [1]:
# PLE_ResNet_Final_Fixed_v13.py
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, GlobalAveragePooling1D, 
                                     Dense, Reshape, Layer, Concatenate, Lambda, Add, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ---------------------- 1. 物理参数定义 (保持不动) ----------------------
TYPE_NAMES = ["SingleTone", "Chirp", "Pulse", "HoppingJam", "NoiseFM", "NoiseAM", "Comb", "Mixed", "Normal"]
TRUTH_MAP = {
    0: [500, 0], 1: [60000, 0], 2: [45000, 0], 3: [85000, 0],
    4: [70000, 0], 5: [35000, 0], 6: [65000, 0], 7: [80000, 0], 8: [25000, 0]
}
FS = 200000

# ---------------------- 2. 核心组件 (PLE & ResNet) ----------------------
class PLELayer(Layer):
    """专家系统：expert_dim 扩充至 256 以应对任务冲突"""
    def __init__(self, num_tasks, num_shared_experts, num_task_experts, expert_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_tasks, self.num_shared_experts, self.num_task_experts, self.expert_dim = num_tasks, num_shared_experts, num_task_experts, expert_dim
    def build(self, input_shape):
        self.shared_experts = [Dense(self.expert_dim, activation='relu') for _ in range(self.num_shared_experts)]
        self.task_experts = [[Dense(self.expert_dim, activation='relu') for _ in range(self.num_task_experts)] for _ in range(self.num_tasks)]
        self.gates = [Dense(self.num_shared_experts + self.num_task_experts, activation='softmax') for _ in range(self.num_tasks)]
    def call(self, inputs):
        shared_outputs = [ex(inputs) for ex in self.shared_experts]
        final_outputs = []
        for t in range(self.num_tasks):
            task_outputs = [ex(inputs) for ex in self.task_experts[t]]
            all_ex = tf.stack(shared_outputs + task_outputs, axis=1)
            gate = tf.expand_dims(self.gates[t](inputs), axis=-1)
            final_outputs.append(tf.reduce_sum(all_ex * gate, axis=1))
        return final_outputs

class ResNetBlock1D(Layer):
    """时域特征残差块"""
    def __init__(self, filters, kernel_size, strides=1, **kwargs):
        super().__init__(**kwargs)
        self.filters, self.kernel_size, self.strides = filters, kernel_size, strides
    def build(self, input_shape):
        self.conv1 = Conv1D(self.filters, self.kernel_size, strides=self.strides, padding='same', use_bias=False)
        self.bn1 = BatchNormalization()
        self.conv2 = Conv1D(self.filters, self.kernel_size, padding='same', use_bias=False)
        self.bn2 = BatchNormalization()
        self.shortcut = Conv1D(self.filters, 1, strides=self.strides, padding='same') if input_shape[-1] != self.filters or self.strides != 1 else Lambda(lambda x: x)
    def call(self, inputs):
        x = tf.nn.relu(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        return tf.nn.relu(Add()([x, self.shortcut(inputs)]))

# ---------------------- 3. 模型构建 (修复 Metrics 命名问题) ----------------------
def build_fixed_resnet_ple(L, num_classes):
    """权重极化且修复指标命名的 PLE 模型"""
    inputs = Input(shape=(L,), name='input_signal')
    
    # 特征分支 (保持三分支逻辑)
    t = ResNetBlock1D(64, 7, strides=2)(Reshape((L, 1))(inputs))
    t = GlobalAveragePooling1D()(t)
    
    def stats_calc(x):
        return tf.concat([tf.reduce_mean(x, 1, True), tf.math.reduce_std(x, 1, True)], 1)
    st = Dense(64, activation='relu')(Lambda(stats_calc)(inputs))
    
    fusion = Concatenate()([t, st])
    ple_out = PLELayer(3, 2, 1, 256)(fusion) # 大专家容量
    
    # 任务输出
    det = Dense(1, activation='sigmoid', name='detection_output')(ple_out[0])
    cls = Dense(num_classes, activation='softmax', name='classification_output')(Dense(128, activation='relu')(ple_out[1]))
    reg = Dense(3, activation='linear', name='reg_clip')(ple_out[2])
    
    model = Model(inputs, [det, cls, reg])
    
    # 核心修正：显式指定 metrics=['accuracy'] 确保生成监测项
    model.compile(optimizer=Adam(learning_rate=2e-4), 
                  loss={'detection_output': 'binary_crossentropy', 
                        'classification_output': 'sparse_categorical_crossentropy', 
                        'reg_clip': 'mse'},
                  loss_weights={'detection_output': 1.0, 
                                'classification_output': 10.0, # 极高权重压制负迁移
                                'reg_clip': 0.2},
                  metrics={'detection_output': 'accuracy',
                           'classification_output': 'accuracy'}) 
    return model

# ---------------------- 4. 数据加载与集成训练逻辑 ----------------------
def load_and_preprocess(root_path):
    all_sig, all_lab, all_jnr = [], [], []
    files = [f for f in os.listdir(root_path) if f.endswith('_X.npy')]
    for fx in files:
        m = re.search(r'jnr(-?\d+)', fx)
        jnr_v = float(m.group(1)) if m else 0.0
        sig = np.load(os.path.join(root_path, fx))
        lab = np.load(os.path.join(root_path, fx.replace('_X.npy', '_Y.npy')))
        if sig.ndim == 3: sig = sig[:, :, 0]
        all_sig.append(sig); all_lab.append(lab); all_jnr.append(np.full(len(lab), jnr_v))
    
    X = np.vstack(all_sig); Y = np.concatenate(all_lab); J = np.concatenate(all_jnr)
    X_n = np.array([StandardScaler().fit_transform(s.reshape(-1, 1)).ravel() for s in X])
    
    p = np.stack([np.array([TRUTH_MAP[l][0]/FS for l in Y]), 
                  np.array([TRUTH_MAP[l][1]/(FS/2) for l in Y]), 
                  np.clip((10**(J/10)-0.1)/(1000-0.1), 0, 1)], axis=1)
    
    return train_test_split(X_n, (Y<8).astype(float), Y, p.astype(np.float32), J, test_size=0.3, stratify=Y)

def main():
    root_dir = "/root/autodl-tmp/validate/0218/dataset_final_ready_v5"
    X_train, X_test, y_det_t, y_det_v, y_type_t, y_type_v, y_param_t, y_param_v, j_t, j_v = load_and_preprocess(root_dir)
    
    ensemble_models = []
    for i in range(3):
        print(f"\n🔥 启动集成模型训练 [{i+1}/3] - 极化权重配置模式...")
        m = build_fixed_resnet_ple(1024, 9)
        
        # 指标修正后的回调函数
        cbs = [EarlyStopping(monitor='val_classification_output_accuracy', patience=8, restore_best_weights=True, mode='max'),
               ReduceLROnPlateau(monitor='val_classification_output_accuracy', factor=0.5, patience=4, mode='max', verbose=1)]
        
        m.fit(X_train, {'detection_output': y_det_t, 'classification_output': y_type_t, 'reg_clip': y_param_t},
              validation_split=0.1, epochs=40, batch_size=128, callbacks=cbs, verbose=1)
        ensemble_models.append(m)

    # 5. 最终报表评估 (集成预测)
    preds_d, preds_c, preds_r = [], [], []
    for m in ensemble_models:
        d, c, r = m.predict(X_test, verbose=0)
        preds_d.append(d); preds_c.append(c); preds_r.append(r)
    
    p_det = (np.mean(preds_d, 0) > 0.5).astype(int).ravel()
    p_cls = np.argmax(np.mean(preds_c, 0), axis=1)
    p_reg = np.mean(preds_r, 0)
    
    print("\n" + "="*60 + "\n📈 收官评估报表 (Metrics 对齐版)\n" + "="*60)
    print(f"📡 干扰检测准确率: {accuracy_score(y_det_v, p_det)*100:.2f}%")
    print(f"🎯 总体识别准确率: {accuracy_score(y_type_v, p_cls)*100:.2f}%")
    
    print("\n📈 分 JNR 识别率统计:")
    for j in sorted(np.unique(j_v)):
        mask = (j_v == j)
        print(f"   🔹 JNR {j:>3} dB: {accuracy_score(y_type_v[mask], p_cls[mask])*100:.2f}%")
    
    maes = mean_absolute_error(y_param_v, p_reg, multioutput='raw_values')
    print(f"\n📝 参数回归误差 (MAE): BW={maes[0]:.4f}, Fc={maes[1]:.4f}, JNR={maes[2]:.4f} [对齐成功]")
    print("="*60)

if __name__ == "__main__":
    main()

2026-02-19 22:24:38.512296: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-19 22:24:38.578890: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-02-19 22:24:39.471270: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT



🔥 启动集成模型训练 [1/3] - 极化权重配置模式...


2026-02-19 22:25:39.577241: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 925 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:39:00.0, compute capability: 8.9


Epoch 1/40


2026-02-19 22:25:43.683001: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2026-02-19 22:25:43.695295: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-02-19 22:25:44.068918: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x7fe630033040 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-19 22:25:44.068964: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): NVIDIA GeForce RTX 4090 D, Compute Capability 8.9
2026-02-19 22:25:44.080152: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-19 22:25:44.293870: I ./tensorflow/compiler/jit/device_compiler.h:180] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the

670/670 [==============================] - 20s 19ms/step - loss: 16.2019 - detection_output_loss: 0.0305 - classification_output_loss: 1.6167 - reg_clip_loss: 0.0219 - detection_output_accuracy: 0.9998 - classification_output_accuracy: 0.4080 - val_loss: 14.1173 - val_detection_output_loss: 1.2966e-04 - val_classification_output_loss: 1.4114 - val_reg_clip_loss: 0.0184 - val_detection_output_accuracy: 1.0000 - val_classification_output_accuracy: 0.4885 - lr: 2.0000e-04
Epoch 2/40
670/670 [==============================] - 12s 17ms/step - loss: 12.7573 - detection_output_loss: 3.8635e-05 - classification_output_loss: 1.2754 - reg_clip_loss: 0.0153 - detection_output_accuracy: 1.0000 - classification_output_accuracy: 0.5520 - val_loss: 12.6250 - val_detection_output_loss: 1.7248e-05 - val_classification_output_loss: 1.2622 - val_reg_clip_loss: 0.0140 - val_detection_output_accuracy: 1.0000 - val_classification_output_accuracy: 0.5565 - lr: 2.0000e-04
Epoch 3/40
670/670 [=================

2026-02-19 22:47:44.790494: W tensorflow/tsl/framework/bfc_allocator.cc:485] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.00MiB (rounded to 4194304)requested by op model_2/res_net_block1d_2/conv1d/Conv1D
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-02-19 22:47:44.790572: I tensorflow/tsl/framework/bfc_allocator.cc:1039] BFCAllocator dump for GPU_0_bfc
2026-02-19 22:47:44.790597: I tensorflow/tsl/framework/bfc_allocator.cc:1046] Bin (256): 	Total Chunks: 284, Chunks in use: 284. 71.0KiB allocated for chunks. 71.0KiB in use in bin. 21.2KiB client-requested in use in bin.
2026-02-19 22:47:44.790613: I tensorflow/tsl/framework/bfc_allocator.cc:1046] Bin (512): 	Total Chunks: 18, Chunks in use: 18. 9.2KiB allocated for chunks. 9.2KiB in use in bin. 9.0KiB client-requested in use in bin.
2026-02-19 22:47:44.79062

ResourceExhaustedError: Graph execution error:

Detected at node 'model_2/res_net_block1d_2/conv1d/Conv1D' defined at (most recent call last):
    File "/root/miniconda3/lib/python3.8/runpy.py", line 194, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/root/miniconda3/lib/python3.8/runpy.py", line 87, in _run_code
      exec(code, run_globals)
    File "/root/miniconda3/lib/python3.8/site-packages/ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "/root/miniconda3/lib/python3.8/site-packages/traitlets/config/application.py", line 1043, in launch_instance
      app.start()
    File "/root/miniconda3/lib/python3.8/site-packages/ipykernel/kernelapp.py", line 725, in start
      self.io_loop.start()
    File "/root/miniconda3/lib/python3.8/site-packages/tornado/platform/asyncio.py", line 215, in start
      self.asyncio_loop.run_forever()
    File "/root/miniconda3/lib/python3.8/asyncio/base_events.py", line 570, in run

In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, mean_absolute_error)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Conv2D, BatchNormalization, MaxPooling1D, MaxPooling2D,
                                     GlobalAveragePooling1D, GlobalAveragePooling2D, Dense, Reshape, Layer,
                                     Concatenate, Lambda, Add, Multiply, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy, SparseCategoricalCrossentropy, MeanSquaredError

# ---------------------- 1. 物理参数与科研指标定义 ----------------------
TYPE_NAMES = ["SingleTone", "Chirp", "Pulse", "HoppingJam", "NoiseFM", "NoiseAM", "Comb", "Mixed", "Normal"]
TRUTH_MAP = {
    0: [500, 0], 1: [60000, 0], 2: [45000, 0], 3: [85000, 0],
    4: [70000, 0], 5: [35000, 0], 6: [65000, 0], 7: [80000, 0], 8: [25000, 0]
}
FS = 200000

# ---------------------- 2. 核心 PLE 与 ResNet 分支组件 (严禁修改) ----------------------
class PLELayer(Layer):
    """专家系统：通过门控网络实现任务间特征的动态剥离"""
    def __init__(self, num_tasks, num_shared_experts, num_task_experts, expert_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_tasks, self.num_shared_experts, self.num_task_experts, self.expert_dim = num_tasks, num_shared_experts, num_task_experts, expert_dim
    def build(self, input_shape):
        self.shared_experts = [Dense(self.expert_dim, activation='relu') for _ in range(self.num_shared_experts)]
        self.task_experts = [[Dense(self.expert_dim, activation='relu') for _ in range(self.num_task_experts)] for _ in range(self.num_tasks)]
        self.gates = [Dense(self.num_shared_experts + self.num_task_experts, activation='softmax') for _ in range(self.num_tasks)]
    def call(self, inputs):
        shared_outputs = [ex(inputs) for ex in self.shared_experts]
        final_outputs = []
        for t in range(self.num_tasks):
            task_outputs = [ex(inputs) for ex in self.task_experts[t]]
            all_ex = tf.stack(shared_outputs + task_outputs, axis=1)
            gate = tf.expand_dims(self.gates[t](inputs), axis=-1)
            final_outputs.append(tf.reduce_sum(all_ex * gate, axis=1))
        return final_outputs

class ResNetBlock1D(Layer):
    """时域残差块：捕获信号的局部跳变特征"""
    def __init__(self, filters, kernel_size, strides=1, **kwargs):
        super().__init__(**kwargs)
        self.filters, self.kernel_size, self.strides = filters, kernel_size, strides
    def build(self, input_shape):
        self.conv1 = Conv1D(self.filters, self.kernel_size, strides=self.strides, padding='same', use_bias=False)
        self.bn1 = BatchNormalization()
        self.conv2 = Conv1D(self.filters, self.kernel_size, padding='same', use_bias=False)
        self.bn2 = BatchNormalization()
        self.shortcut = Conv1D(self.filters, 1, strides=self.strides, padding='same') if input_shape[-1] != self.filters or self.strides != 1 else Lambda(lambda x: x)
    def call(self, inputs):
        x = tf.nn.relu(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        return tf.nn.relu(Add()([x, self.shortcut(inputs)]))

class EfficientTemporalEncoder(Layer):
    """时序编码器：替代 LSTM 以提高实测环境下的推理速度"""
    def __init__(self, **kwargs): super().__init__(**kwargs)
    def build(self, input_shape):
        self.conv = Conv1D(128, 3, padding='same', activation='relu')
        self.pool = GlobalAveragePooling1D()
    def call(self, inputs): return self.pool(self.conv(inputs))

# ---------------------- 3. 数据流水线适配 (适配 v5 npy 文件) ----------------------
def load_v5_dataset(root_path):
    """直接读取 v5 版本纯净版 .npy 文件"""
    all_sig, all_lab, all_jnr = [], [], []
    files = sorted([f for f in os.listdir(root_path) if f.endswith('_X.npy')])
    print(f"📂 正在载入 {len(files)} 个实测数据包...")
    for fx in files:
        match = re.search(r'jnr(-?\d+)', fx)
        jnr_v = float(match.group(1)) if match else 0.0
        sig = np.load(os.path.join(root_path, fx))
        lab = np.load(os.path.join(root_path, fx.replace('_X.npy', '_Y.npy')))
        if sig.ndim == 3: sig = sig[:, :, 0] # 提取实部
        all_sig.append(sig); all_lab.append(lab); all_jnr.append(np.full(len(lab), jnr_v))
    
    X = np.vstack(all_sig); Y = np.concatenate(all_lab); J = np.concatenate(all_jnr)
    # 标准化：实测环境下必须逐样本进行 Z-score 归一化
    X_norm = np.array([StandardScaler().fit_transform(s.reshape(-1, 1)).ravel() for s in X])
    # 物理参数标签对齐
    p_labels = np.stack([np.array([TRUTH_MAP[l][0]/FS for l in Y]), 
                         np.array([TRUTH_MAP[l][1]/(FS/2) for l in Y]), 
                         np.clip((10**(J/10)-0.1)/(1000-0.1), 0, 1)], axis=1)
    
    return train_test_split(X_norm, (Y<8).astype(float), Y, p_labels.astype(np.float32), J, test_size=0.3, stratify=Y)

# ---------------------- 4. 模型构建：权重极化版 (解决负迁移) ----------------------
def build_resnet_ple_v15(L, num_classes):
    """集成三分支特征提取与极化 Loss 配置"""
    inputs = Input(shape=(L,), name='input_signal')
    
    # 1. 时域 ResNet 分支
    t = ResNetBlock1D(64, 7, strides=2)(Reshape((L, 1))(inputs))
    t = GlobalAveragePooling1D()(t)
    
    # 2. 时域统计分支
    def stat_fn(x): return tf.concat([tf.reduce_mean(x, 1, True), tf.math.reduce_std(x, 1, True)], 1)
    st = Dense(64, activation='relu')(Lambda(stat_fn)(inputs))
    
    # 特征融合
    fused = Concatenate()([t, st])
    # 增加专家维度 expert_dim=256 缓解回归对分类的牵制
    ple_out = PLELayer(3, 2, 1, 256)(fused)
    
    # 三任务头
    det = Dense(1, activation='sigmoid', name='detection_output')(ple_out[0])
    cls = Dense(num_classes, activation='softmax', name='classification_output')(Dense(128, activation='relu')(ple_out[1]))
    reg = Dense(3, activation='linear', name='reg_clip')(ple_out[2])
    
    model = Model(inputs, [det, cls, reg])
    # 优化：极化权重配置 (分类权重设为 10.0 以跳出“Mixed”陷阱)
    model.compile(optimizer=Adam(learning_rate=2e-4), 
                  loss={'detection_output': 'binary_crossentropy', 
                        'classification_output': 'sparse_categorical_crossentropy', 
                        'reg_clip': 'mse'},
                  loss_weights={'detection_output': 1.0, 'classification_output': 10.0, 'reg_clip': 0.2},
                  metrics={'detection_output': 'accuracy', 'classification_output': 'accuracy'})
    return model

# ---------------------- 5. 进度统计与多指标评估 ----------------------
def evaluate_final(models, X_test, y_det, y_type, y_param, j_test):
    """集成预测平均与 NRMSE 保护逻辑"""
    all_d, all_c, all_r = [], [], []
    for m in models:
        d, c, r = m.predict(X_test, batch_size=128, verbose=0)
        all_d.append(d); all_c.append(c); all_r.append(r)
    
    p_det, p_cls, p_reg = (np.mean(all_d, 0) > 0.5).astype(int).ravel(), np.argmax(np.mean(all_c, 0), 1), np.mean(all_r, 0)
    
    print("\n" + "="*60 + "\n📊 实测全维度科研评估报表\n" + "="*60)
    print(f"📡 干扰检测率: {accuracy_score(y_det, p_det)*100:.2f}%")
    print(f"🎯 总体识别率: {accuracy_score(y_type, p_cls)*100:.2f}%")
    
    for j in sorted(np.unique(j_test)):
        mask = (j_test == j)
        print(f"   🔹 JNR {j:>3} dB | Accuracy: {accuracy_score(y_type[mask], p_cls[mask])*100:.2f}% [OK]")
    
    maes = mean_absolute_error(y_param, p_reg, multioutput='raw_values')
    print(f"\n📝 物理量误差 MAE: 带宽={maes[0]:.4f}, 频率={maes[1]:.4f}, 强度={maes[2]:.4f}")
    print("="*60)

# ---------------------- 6. 主程序入口 ----------------------
def main():
    data_path = "/root/autodl-tmp/validate/0218/dataset_final_ready_v5" 
    X_t, X_v, d_t, d_v, y_t, y_v, p_t, p_v, j_t, j_v = load_v5_dataset(data_path)
    
    ensemble_list = []
    for i in range(3): # 开启集成学习训练模式
        print(f"\n🔥 启动第 {i+1}/3 个集成模型训练...")
        m = build_resnet_ple_v15(1024, 9)
        cbs = [EarlyStopping(monitor='val_classification_output_accuracy', patience=8, restore_best_weights=True, mode='max'),
               ReduceLROnPlateau(monitor='val_classification_output_accuracy', factor=0.5, patience=4, mode='max', verbose=1)]
        m.fit(X_t, {'detection_output': d_t, 'classification_output': y_t, 'reg_clip': p_t},
              validation_split=0.1, epochs=40, batch_size=128, callbacks=cbs, verbose=1)
        ensemble_list.append(m)

    evaluate_final(ensemble_list, X_v, d_v, y_v, p_v, j_v)

if __name__ == "__main__": main()

2026-02-19 23:42:23.049238: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-19 23:42:23.116220: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-19 23:42:24.006951: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


📂 正在载入 18 个实测数据包...

🔥 启动第 1/3 个集成模型训练...


2026-02-19 23:43:24.767513: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22188 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:3d:00.0, compute capability: 8.9


Epoch 1/40


2026-02-19 23:43:29.279044: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2026-02-19 23:43:29.549907: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-02-19 23:43:29.645399: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x7f173c033030 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-19 23:43:29.645445: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): NVIDIA GeForce RTX 4090 D, Compute Capability 8.9
2026-02-19 23:43:29.655605: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-19 23:43:29.867643: I ./tensorflow/compiler/jit/device_compiler.h:180] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the

670/670 [==============================] - 19s 19ms/step - loss: 16.5132 - detection_output_loss: 0.0228 - classification_output_loss: 1.6486 - reg_clip_loss: 0.0208 - detection_output_accuracy: 1.0000 - classification_output_accuracy: 0.3944 - val_loss: 13.6686 - val_detection_output_loss: 1.5309e-04 - val_classification_output_loss: 1.3664 - val_reg_clip_loss: 0.0198 - val_detection_output_accuracy: 1.0000 - val_classification_output_accuracy: 0.5221 - lr: 2.0000e-04
Epoch 2/40
670/670 [==============================] - 12s 18ms/step - loss: 12.9182 - detection_output_loss: 4.0625e-05 - classification_output_loss: 1.2915 - reg_clip_loss: 0.0156 - detection_output_accuracy: 1.0000 - classification_output_accuracy: 0.5413 - val_loss: 12.3072 - val_detection_output_loss: 2.2331e-05 - val_classification_output_loss: 1.2304 - val_reg_clip_loss: 0.0146 - val_detection_output_accuracy: 1.0000 - val_classification_output_accuracy: 0.5611 - lr: 2.0000e-04
Epoch 3/40
670/670 [=================

KeyboardInterrupt: 